# SSS sSFFS Demo — anytime frontier d vs J
Reproduces reuters 10k-d filter frontier (C++ vs Python) and madelon synthetic wrapper.
Fixed y=25, rho_u=0.2, T auto, single-core.

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))
import matplotlib.pyplot as plt

from src.sss.arff import load_arff
from src.sss.sss import SFFS

# reuters filter frontier (tiny demo: d=5,10,15)
data = load_arff("anc/data/reuters_apte.arff")
frontier = []
for d in [5, 10, 15, 25]:
    sffs = SFFS(
        y=25,
        y_back=12,
        rho_u=0.2,
        tau=0.0,
        warmup_probes=50,
        warmup_card=25,
        delta=5,
        seed=1,
        scaler="void",
    )
    sel = sffs.fit_filter(data, target_d=d)
    print(f"d={d} value={sffs.value_:.4f} evals={sffs.evaluations_} sel={sel[:5]}")
    frontier.append((d, sffs.value_))
plt.plot(*zip(*frontier, strict=False), marker="o")
plt.xlabel("d (subset size)")
plt.ylabel("J (Bhattacharyya)")
plt.title("Reuters anytime frontier (Python sSFFS)")
plt.show()

In [ ]:
# wrapper synthetic frontier (madelon-like)
import sklearn.datasets

X, y = sklearn.datasets.make_classification(
    n_samples=200, n_features=20, n_informative=5, random_state=0
)
for d in [5, 10, 15]:
    sffs = SFFS(y=5, rho_u=0.2, tau=1.0, warmup_probes=20, warmup_card=5, delta=5, seed=1)
    sel = sffs.fit(X, y, d=d)
    print(d, sel, sffs.value_)